## Welcome to Week 6 Day 3: Context Engineering with MCP

People used to talk about Prompt Engineering, but now it's given way to a new Skill "Context Engineering".

Philipp Schmid of Google DeepMind wrote the seminal post about Context Engineering:

https://www.philschmid.de/context-engineering

For this week, we will put Context Engineering into practice, heavily using MCP servers.

1. Long-term memory: a knowledge graph the agent writes to and reads back
2. Web search: fresh information from the live web
3. Agentic RAG: a vector store the agent fills from its own research, then searches
4. Integrations: connecting to a live external service, with a local fallback

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on Windows. The server writes its startup output to stderr, but inside a Windows Jupyter kernel that stream has no real file handle behind it, so the launch fails with an `io.UnsupportedOperation: fileno` error, while Mac and Linux are unaffected.

The fix is to send the server's stderr to the null device, so it always has somewhere real to write. We do that once in the next cell, which lets every cell below use `MCPServerStdio` exactly as the OpenAI Agents SDK documents it. On Mac and Linux it costs nothing beyond keeping the server's startup banner out of the notebook.

In [2]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Part 1: Long-term memory

Our first context source is memory. This is the official knowledge graph server: it stores entities, observations about them, and the relationships between them, and keeps them on disk between runs. We point it at `memory/memory.json`, so you can open that file and read the graph it builds.

That gives an agent something it normally lacks: a memory that outlives the conversation. The agent writes facts as it learns them and reads them back later.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory

In [3]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'

In [4]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-5.4-mini"

In [5]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Got it, Ed — I’ll remember that you’re an LLM engineer, teaching a course on AI Agents, and that MCP is a protocol for connecting agents with tools, resources, and prompt templates.

In [6]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

I know that your name is Ed.

I also have these notes about you:
- You’re an LLM engineer.
- You’re teaching a course about AI Agents.
- Your course includes the MCP protocol.

If you want, I can also tell you what I know about your course specifically.

### Check out the trace

https://platform.openai.com/traces

## Part 2: Web search

A model only knows what it was trained on. Web search is the context source that keeps it current.

We use Tavily, a search API built for agents: it returns clean, ranked results an LLM can use directly. Tavily maintains its own MCP server, so connecting is a one-liner.

This needs a free API key:

1. Sign up at https://www.tavily.com
2. The free tier gives you 1,000 searches a month, with no credit card
3. Copy your API key (it starts with `tvly-`) and add it to your `.env` file:

`TAVILY_API_KEY=tvly-xxxx`

In [7]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

Tavily's server offers several tools (search, extract, crawl, map, research). For this lab we only want plain web search, so we restrict the server to `tavily_search`. The OpenAI Agents SDK lets you hand an agent just the tools you choose with a static tool filter. Curating an agent's tools like this is itself context engineering.

In [8]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-5.4-mini"
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [9]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Here’s a brief update on Amazon (AMZN) as of **2026-09-29**:

- **Recent price action:** AMZN has been trading around **$246–$259** recently, after a strong run earlier in the summer. One recent snapshot put it at **$246.18** with a market cap of about **$2.66T**, while another cited **$258.51** earlier in September. The stock looks **somewhat volatile but still near elevated levels**.

- **Latest news themes:**  
  - **Strong Q2 2026 results:** Amazon posted a big earnings beat, with **EPS of $5.75 vs. $1.81 expected** and revenue of **$200.6B vs. $196.2B expected**.  
  - **AWS and AI remain the core story:** Analysts are still upbeat on Amazon’s cloud and AI opportunity, with AWS growth expectations supporting bullish price targets.  
  - **Heavy spending is the main concern:** Amazon raised **2026 capex guidance to about $220B**, mostly for AI infrastructure. That has created some pressure on the stock because investors are weighing future growth against near-term free-cash-flow strain.

- **Analyst outlook:** Overall sentiment remains **bullish**, with several sources showing **“Strong Buy”** consensus and 12-month targets in the **low $300s**. Some recent targets were around **$315–$335**, with bullish cases higher.

- **Bottom line:**  
  **Near term:** AMZN may stay choppy because the market is sensitive to large AI spending and margin pressure.  
  **Long term:** The outlook is still **positive** if AWS growth continues and Amazon converts AI investment into higher revenue and profits.

If you want, I can also give you a **1-paragraph investor-style thesis** or a **bull vs. bear case** for AMZN.

## Part 3: Agentic RAG

RAG, retrieval augmented generation, means giving a model relevant documents to ground its answer. The usual setup loads documents into a vector store up front. Agentic RAG turns that around: the agent builds the knowledge base itself, deciding what is worth keeping and storing it as it works.

We use the official Qdrant MCP server. Qdrant is a vector database, and the server exposes two tools: one stores a piece of text, the other finds the most relevant stored text for a query. It runs fully locally here. `QDRANT_LOCAL_PATH` keeps everything on disk with no separate database to run, and it embeds text with a local model, so there is no extra API key. The first store or search downloads that small embedding model, so it pauses once on first use.

https://github.com/qdrant/mcp-server-qdrant

We hand one agent both Tavily and Qdrant: it researches a topic on the web, stores what it learns, then answers from its own knowledge base.

In [10]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [11]:
INSTRUCTIONS = """You research topics on the web and build up a knowledge base for later.
When you learn something worth keeping, store it in your knowledge base.
When you are asked what you know, search your knowledge base and answer from it."""

model = "gpt-5.4-mini"

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as search_server:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[search_server, vector_server])
        with trace("research and store"):
            result = await Runner.run(agent, "Research the latest news on Nvidia and store the key facts in your knowledge base.", max_turns=20)
        display(Markdown(result.final_output))

I researched the latest Nvidia news and stored the key facts in my knowledge base.

Key takeaways:
- Nvidia announced a record $150 billion increase to its share buyback program, raising remaining authorization to $235 billion.
- Nvidia has been pushing new AI safety and agent tooling, including an Open Agent Safety Platform and open-source physical AI tools.
- Reuters reported Nvidia bought Hugging Face for $12.93 billion to deepen its role in open-source AI.
- Competitive pressure from Chinese AI chipmakers remains a notable theme.
- Market data in the search results shows Nvidia’s stock and valuation remain very strong, with earnings expected on Nov. 17, 2026.

If you want, I can also give you a concise “what changed in the last 30 days” Nvidia briefing.

The agent below has only the knowledge base, no web search. Whatever it tells us about Nvidia, it is recalling from what the first agent stored. That is the retrieval half of RAG.

In [12]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest on Nvidia?")
    display(Markdown(result.final_output))

Based on my knowledge base, the latest on Nvidia is:

- **Massive buyback expansion:** Nvidia increased its share repurchase authorization by **$150 billion**, bringing total remaining buyback capacity to **$235 billion** and saying it expects to run the program through **fiscal 2028**.
- **AI agent tooling:** Nvidia recently unveiled an **Open Agent Safety Platform** / open-agent tools aimed at monitoring and controlling AI agents.
- **Open-source and robotics push:** Recent Nvidia newsroom items emphasize **open-source physical AI tools**, **agent development platforms**, and themes around **AI infrastructure, robotics, digital twins, and enterprise agents** tied to **GTC 2026 / GTC Taipei**.
- **Hugging Face acquisition:** Reuters reported Nvidia bought **Hugging Face for $12.93 billion** on **2026-09-03** to strengthen its position in open-source AI and the developer ecosystem.
- **China/export pressure:** Nvidia is still facing competitive pressure in China, where local chipmakers and firms like Alibaba are building domestic AI accelerators to reduce reliance on Nvidia.
- **Market position:** Nvidia’s share price has remained strong in 2026, with market cap around **$5.435T** in the latest data in the knowledge base.

If you want, I can also give you a **short “what this means for Nvidia” summary** or break it down by **AI, gaming, data center, and geopolitics**.

### Check out the trace

https://platform.openai.com/traces

## Part 4: Integrations

The last context source is a live external service. Here that is market data from Massive (formerly Polygon.io), a popular financial data provider that publishes its own MCP server.

Setting this up is optional. Massive offers a free API key with no credit card, giving you their real end of day market data:

1. Sign up at https://www.massive.com
2. Create an API key
3. Add it to your `.env` file:

`MASSIVE_API_KEY=xxxx`

If you would rather not sign up, the next cell falls back to a local market server, the same kind of MCP server we built on Day 2, which serves simulated prices so the rest of the lab still works.

In [13]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        # mcp_massive still imports mcp.server.fastmcp, an API the mcp SDK's 2.0.0 release removed.
        # mcp_massive doesn't cap its mcp dependency, so pin uvx to an older, compatible mcp here.
        "args": ["--with", "mcp<2.0.0", "--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    market_params = {"command": "uv", "args": ["run", "-m", "backend.market_server"]}

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

market_tools

[Tool(name='search_endpoints', title=None, description='Search for market data API endpoints and built-in finance functions by natural language query. Use this FIRST to find the right endpoint before calling call_api. Covers stocks, options, forex, crypto, futures, indices, ETFs, and economic data. Pass market to pin results to a specific asset class when you already know it; omit it and the server will infer from the query. Use detail="more" to see query parameter docs needed for building call_api requests.', inputSchema={'additionalProperties': False, 'properties': {'query': {'description': 'Natural language search query for API endpoints', 'minLength': 1, 'title': 'Query', 'type': 'string'}, 'scope': {'anyOf': [{'enum': ['all', 'endpoints', 'functions'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Search scope: "endpoints" for API only, "functions" for local functions only, or "all"/omit for both', 'title': 'Scope'}, 'max_results': {'anyOf': [{'maximum': 2

In [ ]:
from backend.market import massive_api_key, get_plan_tier, REALTIME, DELAYED

live_note = "You have access to live market data tools; use them to look up share prices, trends, technical indicators and fundamentals."
tier_notes = {
    REALTIME: "For the latest share price, use the last trade endpoint /v2/last/trade/{TICKER}.",
    DELAYED: "Your plan gives prices on a 15 minute delay: use the snapshot endpoint /v2/snapshot/locale/us/markets/stocks/tickers/{TICKER}. The last trade and quote endpoints are not available.",
}
end_of_day_note = "Your plan is end of day data only: use /v2/aggs/ticker/{TICKER}/prev for the prior close. Snapshot, last trade and quote endpoints return 403 NOT_AUTHORIZED, so don't call them. You are limited to 5 API calls per minute."

def market_note() -> str:
    if not massive_api_key:
        return "You have access to a market data tool; use your lookup_share_price tool to get the current share price for any symbol."
    try:
        return f"{live_note} {tier_notes.get(get_plan_tier(), end_of_day_note)}"
    except Exception:
        return live_note

In [17]:
instructions = market_note()
model = "gpt-5.4-mini"

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Apple (AAPL) last traded at **$338.40** most recently, based on the prior close data.

## That's it for today

Four MCP servers, four kinds of context: long-term memory, web search, a knowledge base the agent builds itself, and a live integration. Choosing which sources an agent needs, and wiring them in, is much of what context engineering is in practice.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Browse an MCP marketplace like glama.ai/mcp or smithery.ai and add another context source to this notebook, using one of today's four patterns.
            </span>
        </td>
    </tr>
</table>